# Starlink Characterization - WD

---

Crie uma um diretório 'data' na raíz. Coloque a pasta métricas Prometheus dentro desse diretório com o nome 'metricas_prometheus'.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib
import pandas as pd
import seaborn as sns
import numpy

In [2]:
ROOT = Path(os.getcwd())

sys.path.append(str(ROOT))

DIR_DATA = ROOT / 'data'
DIR_OUT  = ROOT / 'output'
DIR_SRC  = ROOT / 'src'
DIR_PROM = DIR_DATA / 'metricas_prometheus'

DIR_DATA.mkdir(exist_ok=True)

STARLINK_PATH = DIR_DATA / 'starlink_wd.xlsx'
NETFLOW_PATH  = DIR_DATA / 'netflow_all.csv'

In [3]:
if not STARLINK_PATH.exists():
    print('Arquivo não encontrado!')
else:
    df_sl = pd.read_excel(STARLINK_PATH)
    print('Dados carregados com sucesso!')

Dados carregados com sucesso!


In [4]:
if not NETFLOW_PATH.exists():
    print('Arquivo não encontrado!')
else:
    df_nf = pd.read_csv(NETFLOW_PATH)
    print('Dados carregados com sucesso!')

Dados carregados com sucesso!


## 1. Tratamento de Dados

In [6]:
from src.data_utils import (
    format_netflow_ts,
    format_starlink_ts
)

In [7]:
# Formatação de timestamps
df_sl = format_starlink_ts(df_sl)
df_nf = format_netflow_ts(df_nf)

In [8]:
df_nf.tail()

,type,sampled,export_sysid,first,last,received,in_packets,in_bytes,proto,tcp_flags,...,icmp_code,src6_addr,dst6_addr,first_utc,last_utc,first_local,last_local,flow_duration_s,bytes_por_fluxo,is_short_flow
1053463,FLOW,1,1,2025-12-04 19:08:39.927,2025-12-04 19:08:39.927,2025-12-04 19:14:01.141,1,68,17,........,...,NaN,NaN,NaN,2025-12-04 19:08:39.927000+00:00,2025-12-04 19:08:39.927000+00:00,2025-12-04 16:08:39.927000-03:00,2025-12-04 16:08:39.927000-03:00,0.000,68,True
1053464,FLOW,1,1,2025-12-04 19:08:39.927,2025-12-04 19:08:39.927,2025-12-04 19:14:01.141,1,68,17,........,...,NaN,NaN,NaN,2025-12-04 19:08:39.927000+00:00,2025-12-04 19:08:39.927000+00:00,2025-12-04 16:08:39.927000-03:00,2025-12-04 16:08:39.927000-03:00,0.000,68,True
1053465,FLOW,1,1,2025-12-04 19:08:58.947,2025-12-04 19:08:58.947,2025-12-04 19:14:01.141,1,96,17,........,...,NaN,2803:9810:6479:8308:661c:67ff:febe:5f8,2620:2d:4000:1::40,2025-12-04 19:08:58.947000+00:00,2025-12-04 19:08:58.947000+00:00,2025-12-04 16:08:58.947000-03:00,2025-12-04 16:08:58.947000-03:00,0.000,96,True
1053466,FLOW,1,1,2025-12-04 19:07:48.889,2025-12-04 19:09:06.201,2025-12-04 19:15:01.200,4,288,58,NaN,...,0.0,fe80::661c:67ff:febe:5f8,fe80::7624:9fff:fe48:c608,2025-12-04 19:07:48.889000+00:00,2025-12-04 19:09:06.201000+00:00,2025-12-04 16:07:48.889000-03:00,2025-12-04 16:09:06.201000-03:00,77.312,288,False
1053467,FLOW,1,1,2025-12-04 19:09:14.138,2025-12-04 19:09:36.473,2025-12-04 19:15:01.200,6,432,58,NaN,...,0.0,2803:9810:6479:8308:661c:67ff:febe:5f8,ff02::1:ff48:c608,2025-12-04 19:09:14.138000+00:00,2025-12-04 19:09:36.473000+00:00,2025-12-04 16:09:14.138000-03:00,2025-12-04 16:09:36.473000-03:00,22.335,432,False


In [9]:
df_sl.tail()

,Unnamed: 0,timestamp,lat,long,alt,velocidade,latencia,direction_azimuth,direction_elevation,is_snr_above_noise_floor,...,downloadBps,uploadBps,pingDropRate,popPingLatencyMs,powerIn,dt_utc,dt_local,elapsed_min,hora,dia
124287,124287,1764796340,-22.906424,-43.133194,31.958849,0.100919,21.449280,152.312500,71.900528,True,...,2051106.90,621142.90,0.0,23.772917,50.363300,2025-12-03 21:12:20+00:00,2025-12-03 18:12:20-03:00,3409.500000,18,03/12
124285,124285,1764796340,-22.906424,-43.133194,31.958849,0.100919,21.449280,152.312500,71.900528,True,...,690936.06,545502.80,0.0,20.432880,44.656425,2025-12-03 21:12:20+00:00,2025-12-03 18:12:20-03:00,3409.500000,18,03/12
124286,124286,1764796340,-22.906424,-43.133194,31.958849,0.100919,21.449280,152.312500,71.900528,True,...,36744550.00,252002.17,0.0,21.870583,35.117382,2025-12-03 21:12:20+00:00,2025-12-03 18:12:20-03:00,3409.500000,18,03/12
124288,124288,1764796340,-22.906424,-43.133194,31.958849,0.100919,21.449280,152.312500,71.900528,True,...,613828.00,110771.57,0.0,24.094563,41.560543,2025-12-03 21:12:20+00:00,2025-12-03 18:12:20-03:00,3409.500000,18,03/12
124289,124289,1764796590,-22.906424,-43.133180,34.528884,0.594326,25.895572,152.575211,71.798805,True,...,814022.10,639455.20,0.0,20.702934,39.511940,2025-12-03 21:16:30+00:00,2025-12-03 18:16:30-03:00,3413.666667,18,03/12
